# Intervention Design

Targeted activation interventions: scale components, steer with directions,
zero/mean ablation sweeps, and progressive ablation to measure model robustness.

## Why This Matters

Steering and intervention intervention design enables controlled modification of model behavior by adding, removing, or modifying specific activation components. These causal interventions go beyond correlation to establish what internal representations actually do.

**Key references:**
- [Turner et al. (2023) "Activation Addition"](https://arxiv.org/abs/2308.10248) — Steering model behavior by adding vectors to activations
- [Li et al. (2023) "Inference-Time Intervention"](https://arxiv.org/abs/2306.03341) — Targeted activation interventions for truthfulness

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.intervention_design import (
    scale_component_effect, add_direction_effect, zero_ablation_sweep,
    mean_ablation_effect, progressive_ablation,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Scale Component Effect

Scale a specific component's output (attention or MLP) and measure the
impact on the model's prediction.

In [ ]:
for comp in ['L0_attn', 'L0_mlp', 'L2_attn', 'L3_mlp']:
    result = scale_component_effect(model, tokens, comp, scale=0.0)
    changed = 'CHANGED' if result['prediction_changed'] else 'same'
    print(f"{comp}: KL={result['kl_divergence']:.4f}, "
          f"conf {result['clean_confidence']:.3f}→{result['scaled_confidence']:.3f} [{changed}]")

## Add Direction Effect

Add a direction vector to the residual stream at a specific layer.
Useful for activation steering experiments.

In [ ]:
direction = jax.random.normal(jax.random.PRNGKey(0), (32,))
for mag in [0.1, 1.0, 5.0, 10.0]:
    result = add_direction_effect(model, tokens, layer=2, direction=direction, magnitude=mag)
    changed = 'CHANGED' if result['prediction_changed'] else 'same'
    print(f"mag={mag:5.1f}: pred {result['clean_prediction']}→{result['steered_prediction']}, "
          f"conf {result['clean_confidence']:.3f}→{result['steered_confidence']:.3f} [{changed}]")

## Zero Ablation Sweep

Zero-ablate each component individually and rank by impact.
Reveals which components are most critical for the prediction.

In [ ]:
result = zero_ablation_sweep(model, tokens)
print(f"Most important: {result['most_important']}")
print(f"Prediction changers: {result['n_prediction_changers']}/{len(result['per_component'])}\n")
for comp in result['per_component']:
    bar = '█' * int(min(comp['kl_divergence'], 10) * 3)
    changed = ' ← CHANGED' if comp['prediction_changed'] else ''
    print(f"  {comp['component']:8s}: KL={comp['kl_divergence']:.4f} {bar}{changed}")

## Mean Ablation Effect

Replace a component's output with its mean across positions.
Less destructive than zero ablation — preserves the average contribution.

In [ ]:
for layer in range(4):
    for comp in ['attn', 'mlp']:
        result = mean_ablation_effect(model, tokens, layer=layer, component=comp)
        changed = 'CHANGED' if result['prediction_changed'] else 'same'
        print(f"  L{layer}_{comp}: KL={result['kl_divergence']:.4f} [{changed}]")

## Progressive Ablation

Progressively ablate layers from last to first.
Shows how many layers the model actually needs for its prediction.

In [ ]:
result = progressive_ablation(model, tokens)
print(f"Clean prediction: token {result['clean_prediction']}")
print(f"Min layers needed: {result['min_layers_needed']}\n")
for step in result['per_step']:
    intact = '✓' if step['prediction_intact'] else '✗'
    print(f"  Ablated {step['n_ablated']} layers (from L{step['last_ablated']}): "
          f"KL={step['kl_divergence']:.4f}, pred={step['prediction']} [{intact}]")